# Model comparison (single split)

The first `TRAIN_FRACTION` of matches (by date) train the models; the rest are held out for evaluation. One fit per model, one honest out-of-sample window — much simpler than walk-forward validation, though noisier.

- **models:** multinomial logistic regression, histogram gradient boosting, random forest
- **feature sets:** team ratings + Elo + form, the same plus market probabilities, market probabilities only
- **baselines:** market-implied probabilities and training-set class frequencies

Paths are resolved from the current working directory, so run the notebook from the project root.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
BASE_DIR = Path.cwd()
FEATURES_PATH = BASE_DIR / "match_features.csv"
OUTPUT_DIR = BASE_DIR
TRAIN_FRACTION = 0.7

## Features

In [ ]:
TEAM_FEATURES = [
    "home_gk_rating",
    "home_def_avg",
    "home_mid_avg",
    "home_att_avg",
    "away_gk_rating",
    "away_def_avg",
    "away_mid_avg",
    "away_att_avg",
    "diff_def",
    "diff_mid",
    "diff_att",
    "home_elo",
    "away_elo",
    "elo_diff",
    "home_form5",
    "away_form5",
]

MARKET_FEATURES = ["prob_home_win", "prob_draw", "prob_away_win"]

FEATURE_SETS = {
    "team_form": TEAM_FEATURES,
    #"team_form+market": TEAM_FEATURES + MARKET_FEATURES,
    #"market_only": MARKET_FEATURES,
}

CLASSES = [0, 1, 2]  # home win, draw, away win

## Models

In [ ]:
def make_models():
    """Return one fresh, untrained pipeline per model."""
    return {
        "logistic": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000)),
        ]),
        "hist_gb": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=7,
                min_samples_leaf=60,
                l2_regularization=5.0,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=20,
                random_state=0,
            )),
        ]),
        "random_forest": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=300,
                max_depth=6,
                min_samples_leaf=20,
                random_state=0,
                n_jobs=-1,
            )),
        ]),
    }

## Scoring helpers

In [ ]:
def brier_score(target, probs):
    onehot = np.eye(len(CLASSES))[target]
    return float(((probs - onehot) ** 2).sum(axis=1).mean())

def market_probabilities(test):
    probs = test[MARKET_FEATURES].to_numpy(dtype=float)
    return probs / probs.sum(axis=1, keepdims=True)

## Load and split

In [ ]:
matches = pd.read_csv(FEATURES_PATH)
matches["date"] = pd.to_datetime(matches["date"])
matches = matches.sort_values(["date", "match_id"]).reset_index(drop=True)

if not 0 < TRAIN_FRACTION < 1:
    raise ValueError("TRAIN_FRACTION must be between 0 and 1.")

# Cut at the first held-out match, then keep whole dates on one side so
# results from the same matchday cannot leak between train and test.
cut_index = int(len(matches) * TRAIN_FRACTION)
split_date = matches.loc[cut_index, "date"]
train = matches[matches["date"] < split_date].reset_index(drop=True)
test = matches[matches["date"] >= split_date].reset_index(drop=True)
target = test["target"].to_numpy()

market_probs = market_probabilities(test)
market_loss = log_loss(target, market_probs, labels=CLASSES)

print(f"Split date: {split_date.date()} (train share: {len(train) / len(matches):.1%})")
print(f"Train: {len(train)} matches, {train['date'].min().date()} to {train['date'].max().date()}")
print(f"Test:  {len(test)} matches, {test['date'].min().date()} to {test['date'].max().date()}")

## Fit and score

In [ ]:
results = []
predictions = []


def record(model_name, feature_set, probs):
    loss = log_loss(target, probs, labels=CLASSES)
    results.append({
        "model": model_name,
        "feature_set": feature_set,
        "n": len(test),
        "log_loss": loss,
        "brier": brier_score(target, probs),
        "accuracy": accuracy_score(target, probs.argmax(axis=1)),
        "logloss_diff_vs_market": loss - market_loss,
    })
    predictions.append(pd.DataFrame({
        "model": model_name,
        "feature_set": feature_set,
        "match_id": test["match_id"].to_numpy(),
        "date": test["date"].to_numpy(),
        "target": target,
        "p_home": probs[:, 0],
        "p_draw": probs[:, 1],
        "p_away": probs[:, 2],
    }))


for feature_set, columns in FEATURE_SETS.items():
    for model_name, model in make_models().items():
        model.fit(train[columns], train["target"])
        if list(model.classes_) != CLASSES:
            raise ValueError(f"{model_name} did not see all three classes")
        record(model_name, feature_set, model.predict_proba(test[columns]))

prior = (
    train["target"]
    .value_counts(normalize=True)
    .reindex(CLASSES, fill_value=0.0)
    .to_numpy()
)
record("market", "market", market_probs)
record("class_prior", "baseline", np.tile(prior, (len(test), 1)))

results = pd.DataFrame(results).sort_values("log_loss").reset_index(drop=True).round(4)
predictions = pd.concat(predictions, ignore_index=True)

## Testing accuarcy when guessing only if confidence is above treshold

In [ ]:
for feature_set, columns in FEATURE_SETS.items():
    for model_name, model in make_models().items():
        model.fit(train[columns], train["target"])
        probs = model.predict_proba(test[columns])
        
        test[f"pred_{model_name}"] = np.argmax(probs, axis=1)
        test[f"conf_{model_name}"] = np.max(probs, axis=1)
        bets = test[test[f"conf_{model_name}"] >= 0.6]
        # Accuarcy when bettings only if you're at least 60% confident
        print(f"Accuarcy ({model_name})")
        print((bets["target"] == bets["pred_logistic"]).mean())

In [ ]:
bets = matches[np.max(market_probabilities(matches), axis=1) >= 0.6]

# Accuarcy when bettings only if you're at least 60% confident
print("Accuarcy (market)")
(np.argmax(market_probabilities(bets), axis=1) == bets["target"]).mean()

## Results

In [ ]:
print(f"Home-always accuracy: {(target == 0).mean() * 100:.2f}%")

results

## Confusion matrices

The same test-set matrices as a small-multiples heatmap. Rows are actual outcomes, columns are predictions, and colour shows the share of each actual class (row-normalized); cell labels give the count and percentage.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb",
    "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b",
    "text.color": "#0b0b0b",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
})

INK = "#0b0b0b"
MUTED = "#52514e"
LABELS = ["home", "draw", "away"]


def grid_shape(n):
    """Return a near-square grid for n panels, capped at four columns."""
    ncols = min(4, max(1, n))
    nrows = max(1, (n + ncols - 1) // ncols)
    return nrows, ncols


def plot_confusion_matrices(predictions, results):
    order = [(str(m), str(f)) for m, f in results[["model", "feature_set"]].itertuples(index=False)]
    if not order:
        print("No configurations to plot.")
        return

    counts_by_key = {}
    for (model, feature_set), frame in predictions.groupby(["model", "feature_set"], sort=False):
        actual = frame["target"].to_numpy()
        predicted = frame[["p_home", "p_draw", "p_away"]].to_numpy().argmax(axis=1)
        counts = np.zeros((3, 3), dtype=int)
        for a, p in zip(actual, predicted):
            counts[a, p] += 1
        counts_by_key[(str(model), str(feature_set))] = counts

    n = len(order)
    nrows, ncols = grid_shape(n)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4.1 * ncols, 3.1 * nrows),
        sharex=True, sharey=True, layout="constrained",
    )
    axes = np.atleast_1d(axes).ravel()
    cmap = plt.get_cmap("Blues")

    for ax, key in zip(axes, order):
        counts = counts_by_key[key]
        rates = counts / counts.sum(axis=1, keepdims=True)
        ax.imshow(rates, cmap=cmap, vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                count = counts[i, j]
                rate = rates[i, j]
                color = "white" if rate > 0.55 else "#1f2933"
                text = f"{count:,}" + (f"\n{rate:.0%}" if count else "")
                ax.text(j, i, text, ha="center", va="center", fontsize=9, color=color)
        ax.set_xticks(range(3), LABELS, fontsize=8)
        ax.set_yticks(range(3), LABELS, fontsize=8)
        ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
        ax.set_yticks(np.arange(-0.5, 3, 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=2)
        ax.tick_params(which="both", length=0)
        ax.set_title(f"{key[0]} · {key[1]}", fontsize=9.5, pad=6)

    for ax in axes[n:]:
        ax.axis("off")
    if len(axes) > n:
        note = axes[n]
        note.text(0.02, 0.95, "How to read", fontsize=11, fontweight="bold",
                  va="top", color=INK, transform=note.transAxes)
        note.text(
            0.02, 0.80,
            "Rows: actual outcome\nColumns: predicted outcome\n"
            "Colour: share of the actual class\nLabels: count and row share",
            fontsize=9, va="top", color=MUTED, linespacing=1.6,
            transform=note.transAxes,
        )

    scaler = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
    scaler.set_array([])
    colorbar = fig.colorbar(scaler, ax=axes.tolist(), fraction=0.02, pad=0.02)
    colorbar.set_label("Share of actual outcome (row-normalized)", fontsize=9)
    colorbar.ax.tick_params(labelsize=8)

    fig.suptitle(
        "Confusion matrices: where each model is right and wrong",
        fontsize=15, fontweight="bold", x=0.01, ha="left",
    )
    plt.show()


plot_confusion_matrices(predictions, results)

## Confidence distributions

For each configuration: distribution of the model's confidence (the maximum class probability, so between 1/3 and 1), split by whether the argmax prediction was correct (blue) or wrong (orange). Each histogram is normalized within its own group, so bar heights show the share of that group's predictions in each confidence bin and each distribution sums to 100%.

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

CORRECT_COLOR = "#2a78d6"
WRONG_COLOR = "#eb6834"


def plot_confidence_distributions(predictions, results):
    order = [(str(m), str(f)) for m, f in results[["model", "feature_set"]].itertuples(index=False)]
    if not order:
        print("No configurations to plot.")
        return

    data = {}
    for (model, feature_set), frame in predictions.groupby(["model", "feature_set"], sort=False):
        probs = frame[["p_home", "p_draw", "p_away"]].to_numpy()
        confidence = probs.max(axis=1)
        correct = probs.argmax(axis=1) == frame["target"].to_numpy()
        data[(str(model), str(feature_set))] = (confidence, correct)

    all_confidence = np.concatenate([values[0] for values in data.values()])
    bins = np.linspace(min(1 / 3, all_confidence.min()), 1.0, 26)

    n = len(order)
    nrows, ncols = grid_shape(n)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4.1 * ncols, 3.0 * nrows),
        sharex=True, layout="constrained",
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, key in zip(axes, order):
        confidence, correct = data[key]
        if correct.any():
            ax.hist(
                confidence[correct], bins=bins,
                weights=np.full(correct.sum(), 1.0 / correct.sum()),
                color=CORRECT_COLOR, histtype="step", linewidth=2,
            )
        if (~correct).any():
            ax.hist(
                confidence[~correct], bins=bins,
                weights=np.full((~correct).sum(), 1.0 / (~correct).sum()),
                color=WRONG_COLOR, histtype="step", linewidth=2,
            )
        ax.set_title(f"{key[0]} · {key[1]}", fontsize=9.5, pad=6)
        ax.tick_params(axis="both", labelsize=8, length=0)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
        ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
        ax.set_axisbelow(True)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    for ax in axes[n:]:
        ax.axis("off")

    handles = [
        Line2D([0], [0], color=CORRECT_COLOR, linewidth=2, label="Correct"),
        Line2D([0], [0], color=WRONG_COLOR, linewidth=2, label="Wrong"),
    ]
    axes[0].legend(handles=handles, frameon=False, fontsize=8, loc="upper right")

    fig.supxlabel("Confidence (probability of the predicted outcome)", fontsize=10, color=INK)
    fig.supylabel("Share of predictions", fontsize=10, color=INK)
    fig.suptitle(
        "Confidence distributions: correct vs wrong predictions",
        fontsize=15, fontweight="bold", x=0.01, ha="left",
    )
    plt.show()


plot_confidence_distributions(predictions, results)

In [ ]:
ODDS_PATH = BASE_DIR / "football_matches_odds.csv"
odds = pd.read_csv(ODDS_PATH)[["match_id", "odds_home_win", "odds_draw", "odds_away_win"]]

bets = predictions.merge(odds, on="match_id", validate="many_to_one")
probabilities = bets[["p_home", "p_draw", "p_away"]].to_numpy()
bets["pick"] = probabilities.argmax(axis=1)
bets["confidence"] = probabilities.max(axis=1)
bets["p_pick"] = probabilities[np.arange(len(bets)), bets["pick"]]
bets["odds_taken"] = np.select(
    [bets["pick"] == 0, bets["pick"] == 1, bets["pick"] == 2],
    [bets["odds_home_win"], bets["odds_draw"], bets["odds_away_win"]],
)
bets["implied"] = 1 / bets["odds_taken"]
bets["edge"] = bets["p_pick"] - bets["implied"]
bets["won"] = bets["pick"] == bets["target"]
bets["profit"] = np.where(bets["won"], bets["odds_taken"] - 1, -1.0)
bets["date"] = pd.to_datetime(bets["date"])
bets = bets.sort_values(["date", "match_id"]).reset_index(drop=True)


def summarize(selection):
    """Flat 1-unit stake metrics for a filtered bet table."""
    selection = selection.sort_values(["date", "match_id"])
    if selection.empty:
        return {
            "bets": 0, "hit_rate": np.nan, "avg_odds": np.nan,
            "roi": np.nan, "profit": 0.0, "max_drawdown": 0.0,
        }
    cumulative = np.cumsum(selection["profit"].to_numpy())
    drawdown = cumulative - np.maximum.accumulate(cumulative)
    return {
        "bets": len(selection),
        "hit_rate": float(selection["won"].mean()),
        "avg_odds": float(selection["odds_taken"].mean()),
        "roi": float(selection["profit"].mean()),
        "profit": float(selection["profit"].sum()),
        "max_drawdown": float(-drawdown.min()),
    }


CONFIDENCE_THRESHOLDS = [0.34, 0.40, 0.45, 0.50, 0.55, 0.60]
EDGE_THRESHOLDS = [None, 0.0, 0.02, 0.05]

grid_rows = []
for (model, feature_set), group in bets.groupby(["model", "feature_set"], sort=False):
    for confidence_min in CONFIDENCE_THRESHOLDS:
        for edge_min in EDGE_THRESHOLDS:
            selection = group[group["confidence"] >= confidence_min]
            if edge_min is not None:
                selection = selection[selection["edge"] >= edge_min]
            grid_rows.append({
                "model": model,
                "feature_set": feature_set,
                "min_confidence": confidence_min,
                "min_edge": edge_min,
                **summarize(selection),
            })

strategy_grid = pd.DataFrame(grid_rows)
strategy_grid["hit_rate"] = (strategy_grid["hit_rate"] * 100).round(1)
strategy_grid["roi"] = (strategy_grid["roi"] * 100).round(2)
strategy_grid["avg_odds"] = strategy_grid["avg_odds"].round(2)
strategy_grid["max_drawdown"] = strategy_grid["max_drawdown"].round(1)
strategy_grid = strategy_grid.sort_values("roi", ascending=False).reset_index(drop=True)
strategy_grid.query("bets >= 100").head(30)

### Equity curves

Cumulative flat-stake profit for every configuration under the filter below. Change `EQ_CONFIDENCE` or `EQ_EDGE` to test another strategy; the grid adapts to however many configurations are present.

In [ ]:
EQ_CONFIDENCE = 0.50
EQ_EDGE = None  # e.g. 0.02

selection_rows = []
for (model, feature_set), group in bets.groupby(["model", "feature_set"], sort=False):
    selection = group[group["confidence"] >= EQ_CONFIDENCE]
    if EQ_EDGE is not None:
        selection = selection[selection["edge"] >= EQ_EDGE]
    selection_rows.append((str(model), str(feature_set), selection))

n = len(selection_rows)
if n == 0:
    print("No configurations to plot.")
else:
    nrows, ncols = grid_shape(n)
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4.1 * ncols, 3.0 * nrows), layout="constrained"
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, (model, feature_set, selection) in zip(axes, selection_rows):
        ax.axhline(0, color="#c3c2b7", linewidth=1)
        if selection.empty:
            ax.text(0.5, 0.5, "No bets", ha="center", va="center",
                    transform=ax.transAxes, color=MUTED, fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            cumulative = np.cumsum(selection["profit"].to_numpy())
            ax.plot(np.arange(1, len(cumulative) + 1), cumulative,
                    color="#2a78d6", linewidth=2)
            stats = summarize(selection)
            ax.text(0.03, 0.95, f"{stats['bets']} bets · ROI {stats['roi']:+.1%}",
                    transform=ax.transAxes, va="top", fontsize=8, color=MUTED,
                    bbox=dict(facecolor="#fcfcfb", edgecolor="none", pad=2, alpha=0.85))
        ax.set_title(f"{model} · {feature_set}", fontsize=9.5, pad=6)
        ax.tick_params(axis="both", labelsize=8, length=0)
        ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
        ax.set_axisbelow(True)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    for ax in axes[n:]:
        ax.axis("off")

    filter_label = f"confidence ≥ {EQ_CONFIDENCE:.0%}"
    if EQ_EDGE is not None:
        filter_label += f" · edge ≥ {EQ_EDGE:.0%}"
    fig.supxlabel("Bet number", fontsize=10, color=INK)
    fig.supylabel("Cumulative profit (units)", fontsize=10, color=INK)
    fig.suptitle(
        f"Equity curves — {filter_label}",
        fontsize=15, fontweight="bold", x=0.01, ha="left",
    )
    plt.show()